# 🌿 Harvest Savior — CNN Training Notebook
### AI-Based Crop Disease Detection | GRIET Academic Project
---
**HOW TO USE THIS NOTEBOOK:**
1. Upload this file to [colab.research.google.com](https://colab.research.google.com)
2. Go to **Runtime → Change runtime type → T4 GPU → Save**
3. Run each cell **in order** from top to bottom (Shift+Enter)
4. At the end, download `crop_disease_cnn.h5` and `class_names.json` and place them in `harvest-savior-ai/model/`

> Your PlantVillage folder must be in Google Drive (e.g. `My Drive/PlantVillage/`)


## Step 1 — Mount Google Drive and Verify Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── CHANGE THIS if your PlantVillage folder is somewhere else in your Drive ──
DATASET_ROOT = '/content/drive/MyDrive/PlantVillage'

# Verify the folder exists and show class count
if not os.path.isdir(DATASET_ROOT):
    raise FileNotFoundError(
        f"Could not find dataset at: {DATASET_ROOT}\n"
        "Make sure 'PlantVillage' folder is directly inside 'My Drive'."
    )

classes = sorted([d for d in os.listdir(DATASET_ROOT)
                  if os.path.isdir(os.path.join(DATASET_ROOT, d))])

print(f"✅ Dataset found at: {DATASET_ROOT}")
print(f"   Number of classes : {len(classes)}")
print(f"\n   Classes detected:")
for c in classes:
    print(f"   • {c}")


## Step 2 — Dataset Statistics

In [ ]:
total_images = 0
class_counts = {}

for cls in classes:
    cls_path = os.path.join(DATASET_ROOT, cls)
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    class_counts[cls] = len(imgs)
    total_images += len(imgs)

print(f"{'Class':<55} {'Images':>7}")
print("-" * 64)
for cls, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    bar = "█" * (count // 100)
    print(f"{cls:<55} {count:>7}  {bar}")

print("-" * 64)
print(f"{'TOTAL':<55} {total_images:>7}")


## Step 3 — Visualise Sample Images

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random

# Show one sample image from each of the first 15 classes
sample_classes = classes[:15]
cols = 5
rows = -(-len(sample_classes) // cols)   # ceiling division

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 3.5))
axes = axes.flatten()

for i, cls in enumerate(sample_classes):
    cls_path = os.path.join(DATASET_ROOT, cls)
    imgs = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    img_path = os.path.join(cls_path, random.choice(imgs))
    img = mpimg.imread(img_path)
    axes[i].imshow(img)
    # Wrap long class names for readability
    label = cls.replace('_', ' ').replace('  ', '\n')
    axes[i].set_title(label, fontsize=8, pad=4)
    axes[i].axis('off')

# Hide unused axes
for j in range(len(sample_classes), len(axes)):
    axes[j].axis('off')

plt.suptitle('PlantVillage — One Sample Per Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/sample_images.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Sample grid saved to /content/sample_images.png")


## Step 4 — Install Dependencies and Build tf.data Pipeline

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import json

print(f"✅ TensorFlow {tf.__version__}")
print(f"   GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

# ── Configuration ─────────────────────────────────────────────────────────────
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
VAL_SPLIT   = 0.2
SEED        = 42

# ── Load training set (80%) with augmentation ─────────────────────────────────
# image_dataset_from_directory() scans DATASET_ROOT, assigns one integer label
# per sub-folder (alphabetical order), resizes every image to IMG_SIZE, and
# groups them into batches of BATCH_SIZE.
train_ds = keras.utils.image_dataset_from_directory(
    DATASET_ROOT,
    validation_split=VAL_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',     # one-hot labels required for softmax
)

# ── Load validation set (20%) ─────────────────────────────────────────────────
val_ds = keras.utils.image_dataset_from_directory(
    DATASET_ROOT,
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)
print(f"\n   Classes : {NUM_CLASSES}")
print(f"   Train   : {len(train_ds)} batches")
print(f"   Val     : {len(val_ds)} batches")

# ── Augmentation ──────────────────────────────────────────────────────────────
# Only applied to training images to simulate real-world photo variation.
# WHY AUGMENTATION? Farmers take photos at odd angles, different distances,
# and with varying lighting. Augmentation teaches the model to handle this.
augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1),       # ±10% rotation
    keras.layers.RandomZoom(0.1),           # ±10% zoom
], name="augmentation")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(
    lambda x, y: (augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

val_ds = val_ds.prefetch(AUTOTUNE)

print("\n✅ Data pipeline ready with augmentation.")


## Step 5 — Build Model (MobileNetV2 Transfer Learning)

In [ ]:
# ── Why MobileNetV2? ──────────────────────────────────────────────────────────
# MobileNetV2 was designed for mobile and embedded devices. It has ~3.4M params
# (lightweight), runs fast on low-end hardware, yet achieves excellent accuracy.
# It was pre-trained on ImageNet (14M images), so it already understands
# basic visual features like edges, textures and shapes. We only need to teach
# it the difference between a healthy Tomato leaf and an Early Blight one.
#
# TRANSFER LEARNING — Phase A (frozen base):
#   We freeze all MobileNetV2 layers and only train the new Dense head.
#   This is fast (few parameters to update) and safe (won't overwrite ImageNet knowledge).
#
# TRANSFER LEARNING — Phase B (fine-tuning):
#   We unfreeze the top 30 MobileNetV2 layers and retrain at a very low learning
#   rate, specialising the features for plant disease patterns.

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

# Load MobileNetV2 WITHOUT the top (classification) layer
base_model = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'        # downloads pre-trained weights automatically
)
base_model.trainable = False  # Phase A: freeze entire base

inputs  = keras.Input(shape=(224, 224, 3))
x = preprocess_input(inputs)          # normalises [0,255] → [-1, 1] internally
x = base_model(x, training=False)     # feature extraction (frozen)
x = keras.layers.GlobalAveragePooling2D()(x)   # flatten (7×7×1280) → (1280,)
x = keras.layers.Dropout(0.2)(x)               # regularisation
outputs = keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs, name="HarvestSavior_MobileNetV2")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(model.summary())
print(f"\n✅ Model ready | Classes: {NUM_CLASSES} | Trainable params: {model.count_params():,}")


## Step 6 — Train Phase A (Classification Head Only)

In [ ]:
import datetime

callbacks_a = [
    # Stop training early if validation accuracy doesn't improve for 5 epochs.
    # restore_best_weights=True automatically reloads the best checkpoint.
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    # Save the best model directly to Google Drive — survives runtime disconnects
    keras.callbacks.ModelCheckpoint(
        filepath='/content/drive/MyDrive/crop_disease_cnn.h5',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    # Halve the learning rate if val_loss doesn't improve for 3 epochs
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=1
    ),
]

print("🚀 Phase A training started — base model FROZEN, head only")
start = datetime.datetime.now()

history_a = model.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    callbacks=callbacks_a,
)

elapsed = datetime.datetime.now() - start
print(f"\n✅ Phase A done in {int(elapsed.total_seconds()//60)}m {int(elapsed.total_seconds()%60)}s")
print(f"   Best val accuracy: {max(history_a.history['val_accuracy'])*100:.2f}%")


## ⚡ CRASH RECOVERY — Run this if the runtime disconnected before Phase B
> **If Phase A already completed and the model was saved, run this cell to restore all variables, then skip straight to Step 7 (Phase B).**  
> If your runtime is still alive from Phase A, skip this cell entirely.

In [ ]:
import os, json, datetime
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# ── Step 1: Remount Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)   # skips prompt if already mounted

DATASET_ROOT = '/content/drive/MyDrive/PlantVillage'

# ── Step 2: Rebuild the data pipeline ────────────────────────────────────────
# We must recreate train_ds / val_ds because they are lost on reconnect.
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
VAL_SPLIT  = 0.2
SEED       = 42

train_ds = keras.utils.image_dataset_from_directory(
    DATASET_ROOT, validation_split=VAL_SPLIT, subset="training",
    seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical',
)
val_ds = keras.utils.image_dataset_from_directory(
    DATASET_ROOT, validation_split=VAL_SPLIT, subset="validation",
    seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical',
)
CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)

augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),
], name="augmentation")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(
    lambda x, y: (augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

print(f"✅ Data pipeline rebuilt | {NUM_CLASSES} classes")

# ── Step 3: Load the saved model from Drive (persistent across disconnects) ──
MODEL_PATH = '/content/drive/MyDrive/crop_disease_cnn.h5'
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        "crop_disease_cnn.h5 not found in Google Drive.\n"
        "Unfortunately you will need to re-run Phase A (Step 6).\n"
        "The model will now save to Drive automatically so this won't happen again."
    )

model = keras.models.load_model(MODEL_PATH)
print(f"✅ Model loaded from {MODEL_PATH}")
print(f"   Val accuracy at load: checking...")
loss, acc = model.evaluate(val_ds, verbose=0)
print(f"   Loss={loss:.4f}  Accuracy={acc*100:.2f}%")

# ── Step 4: Reconstruct base_model reference ──────────────────────────────────
# The loaded model contains MobileNetV2 as a sub-layer. We extract it by name
# so that Phase B can unfreeze its top layers for fine-tuning.
base_model = None
for layer in model.layers:
    if 'mobilenetv2' in layer.name.lower():
        base_model = layer
        break

if base_model is None:
    raise RuntimeError(
        "Could not find MobileNetV2 layer inside the loaded model.\n"
        "Layer names found: " + str([l.name for l in model.layers])
    )

print(f"✅ base_model extracted: '{base_model.name}' ({len(base_model.layers)} layers)")
print()
print("🚀 All variables restored. Scroll down and run Step 7 (Phase B Fine-Tuning).")


## Step 7 — Train Phase B (Fine-Tuning Top Layers)

In [ ]:
# Unfreeze the top 30 layers of MobileNetV2 for fine-tuning.
# WHY only 30? The early layers of any CNN detect generic patterns (edges,
# gradients). Those are already perfect from ImageNet training and we don't
# want to change them. Only the later layers detect high-level patterns which
# we want to specialise for plant disease leaf textures.
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

frozen   = sum(1 for l in base_model.layers if not l.trainable)
unfrozen = sum(1 for l in base_model.layers if l.trainable)
print(f"   Frozen: {frozen} layers  |  Unfrozen: {unfrozen} layers")

# IMPORTANT: recompile at 10× lower learning rate.
# If we use a high LR here, the pre-trained weights get destroyed after one step.
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_b = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath='/content/drive/MyDrive/crop_disease_cnn.h5',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
]

print("🚀 Phase B fine-tuning started")
start = datetime.datetime.now()

history_b = model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    callbacks=callbacks_b,
)

elapsed = datetime.datetime.now() - start
print(f"\n✅ Phase B done in {int(elapsed.total_seconds()//60)}m {int(elapsed.total_seconds()%60)}s")
print(f"   Best val accuracy: {max(history_b.history['val_accuracy'])*100:.2f}%")


## Step 8 — Evaluate and Plot Training Curves

In [ ]:
# ── Final evaluation ──────────────────────────────────────────────────────────
loss, accuracy = model.evaluate(val_ds, verbose=1)
print(f"\n   Final Validation Loss     : {loss:.4f}")
print(f"   Final Validation Accuracy : {accuracy*100:.2f}%")

# ── Plot accuracy and loss curves ─────────────────────────────────────────────
acc     = history_a.history['accuracy']     + history_b.history['accuracy']
val_acc = history_a.history['val_accuracy'] + history_b.history['val_accuracy']
loss_h  = history_a.history['loss']         + history_b.history['loss']
val_loss= history_a.history['val_loss']     + history_b.history['val_loss']
fine_tune_start = len(history_a.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(acc,     label='Train Accuracy',      color='#2e7d32')
axes[0].plot(val_acc, label='Validation Accuracy', color='#81c784', linestyle='--')
axes[0].axvline(fine_tune_start, color='grey', linestyle=':', label='Fine-tune start')
axes[0].set_title('Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(loss_h,  label='Train Loss',      color='#c62828')
axes[1].plot(val_loss,label='Validation Loss', color='#ef9a9a', linestyle='--')
axes[1].axvline(fine_tune_start, color='grey', linestyle=':', label='Fine-tune start')
axes[1].set_title('Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Harvest Savior CNN — Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/training_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved to /content/training_plots.png")


## Step 9 — Save class_names.json and Download Both Files

In [ ]:
from google.colab import files

# ── Save class names in the same order used during training ───────────────────
# predictor.py reads this file to map argmax index → disease label.
# The order MUST match the alphabetical sub-folder order used by
# image_dataset_from_directory() — which is exactly what CLASS_NAMES contains.
# Save class_names.json to Drive so it also survives disconnects
with open('/content/drive/MyDrive/class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)

print("✅ Both files saved to your Google Drive (My Drive):")
print(f"   crop_disease_cnn.h5   — trained model")
print(f"   class_names.json      — class label list")
print()
print("Downloading to your laptop now...")
print("After downloading, place BOTH files in:")
print("  HarvestSavior\\harvest-savior-ai\\model\\")
print()
print("Then restart Flask — it will auto-detect the model and switch to REAL MODE.")
print()

# Trigger browser download dialogs
files.download('/content/drive/MyDrive/crop_disease_cnn.h5')

files.download('/content/drive/MyDrive/class_names.json')